In [0]:
from pyspark.sql.functions import(col, concat, lit, to_timestamp, to_date,try_to_timestamp, concat_ws,trim, upper, lower )

In [0]:
%run "../includes/common_functions"


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1.silver;

In [0]:
lap_times_df = spark.read.table("f1.bronze.lap_times")

In [0]:
lap_times_renamed_df = lap_times_df \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumnRenamed("milliseconds", "lap_time_milliseconds")

In [0]:
lap_times_renamed_df = lap_times_renamed_df \
    .withColumn("time", trim(col("time")))

In [0]:
lap_times_clean_df = lap_times_renamed_df \
    .filter(col("race_id").isNotNull()) \
    .filter(col("driver_id").isNotNull()) \
    .dropDuplicates(["race_id", "driver_id", "lap"])

In [0]:
lap_times_final_df = add_ingestion_date(lap_times_clean_df) \
    .withColumn("environment", lit("production")) \
    .withColumn("file_date", to_date(lit("2026-05-07")))

In [0]:
lap_times_final_df.createOrReplaceTempView("lap_times_updates")

In [0]:
if spark.catalog.tableExists("f1.silver.lap_times"):

    spark.sql("""

    MERGE INTO f1.silver.lap_times tgt
    USING lap_times_updates src
    ON tgt.race_id = src.race_id
    AND tgt.driver_id = src.driver_id
    AND tgt.lap = src.lap

    WHEN MATCHED THEN
    UPDATE SET *

    WHEN NOT MATCHED THEN
    INSERT *

    """)

else:

    lap_times_final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("f1.silver.lap_times")

In [0]:
%sql

SELECT *
FROM f1.silver.lap_times
LIMIT 10;